In [ ]:
# 0myStrategy/Covered Call.ipynb
# -*- coding: utf-8 -*-

import sys
import os
from pathlib import Path
notebook_path = os.path.abspath('')  # مسیر فعلی
root_dir = Path(notebook_path).parent
sys.path.append(str(root_dir))

from data.downloader import MarketDownloader  # noqa: E402
from data.cleaner import DataCleaner  # noqa: E402
import pandas as pd  # noqa: E402
import numpy as np  # noqa: E402
from scipy.stats import norm  # noqa: E402

df_raw =  MarketDownloader.from_tsetmc_direct()
df_cleaned = DataCleaner.clean(df_raw)
df_final = DataCleaner.add_derived_columns(df_cleaned)
df_final.head(2)


,Ticker,Name,StrikePrice,UnderlyingTicker,UnderlyingPrice,MaturityDate,DaysToMaturity,OpenPositions,Volume,Value,...,AskVolume,InstrumentCode,InstrumentCode-UA,IntrinsicValue,MidPrice,TimeValue,Moneyness,OptionStatus,SpreadPct,PremiumOverIntrinsic
3,ضملت5028,اختيارخ وبملت-900-1405/05/20,900,وبملت,1106,1405-05-20,24,45045,5379,5.949174e+09,...,89,11227468399461524,778253364357513,206.0,193.5,0.0,1.228889,ITM,0.067183,0.912621
4,ضملت6016,اختيارخ وبملت-800-1405/06/18,800,وبملت,1106,1405-06-18,53,101767,2807,3.094588e+09,...,100,52956206355211864,778253364357513,306.0,335.0,29.0,1.382500,ITM,0.023881,1.111111
5,ضملت6019,اختيارخ وبملت-1100-1405/06/18,1100,وبملت,1106,1405-06-18,53,1008737,363726,4.022810e+11,...,6056,6266410264554948,778253364357513,6.0,100.5,94.5,1.005455,ITM,0.009950,16.666667


In [12]:
def covered_call_with_fees(ticker, premium_call, stock_price, strike_price, contract_size,
                           opt_sell_commission, stock_buy_commission, exercise_fee_rate, exercise_tax_rate, days):

    # ====================== 1. کارمزدهای ورود (همیشه اعمال می‌شوند) ======================
    # کارمزد فروش اختیار
    option_fee = -round(premium_call * contract_size * opt_sell_commission, 0)
    # کارمزد خرید سهام
    stock_buy_fee = -round(stock_price * contract_size * stock_buy_commission, 0)
     # کل کارمزد ورود به استراتژی
    entry_fees = option_fee + stock_buy_fee

    # ====================== 2. کارمزدهای خروج/اعمال  ======================
    exercise_fee = -round(strike_price * contract_size * exercise_fee_rate, 0)
    # مالیات اعمال
    exercise_tax = -round(strike_price * contract_size * exercise_tax_rate, 0)

    # ====================== 3. مبالغ اصلی ======================
    premium_received = premium_call * contract_size     # دریافتی از فروش اختیار
    stock_cost = -stock_price * contract_size           # ارزش خرید سهام

    # ====================== 4. سرمایه اولیه خالص ======================
    # سرمایه خالص درگیر (جریان نقدی اولیه)
    net_investment = stock_cost + premium_received + entry_fees

    # تحویل سهام در قیمت اعمال (دریافت وجه)
    strike_received = strike_price * contract_size

    # تحویل سهام در قیمت اعمال (دریافت وجه خالص)
    net_received = strike_received + exercise_fee + exercise_tax

    # ====================== 5. سود خالص ======================
    net_profit = net_received + net_investment

    # ====================== 6. درصد بازده ======================
    profit_percent = (round((net_profit / abs(net_investment)) * 100, 2) if net_investment != 0 else 0)
    monthly_return = round(profit_percent * (30 / days), 2)

    # ====================== 7. قیمت سربه‌سر (فقط سناریوی اصلی) ======================
    downside_protection = premium_received + entry_fees + exercise_fee + exercise_tax
    
    # قیمتی که در آن سرمایه اولیه جبران شود
    break_even_price = round(stock_price - (downside_protection / contract_size), 0)

    # ====================== 8. درصد افت مجاز (فقط سناریوی اصلی) ======================
    max_drop_percent = round(((stock_price - break_even_price) / stock_price) * 100, 2)

    return {
    'net_profit': net_profit,
    'monthly_return': monthly_return,
    'break_even_price': break_even_price,
    'max_drop_percent': max_drop_percent}

In [13]:
filter_option = df_final[
    (df_final['DaysToMaturity'] > 2.0) &
    (df_final['Type'].apply(lambda x: x.name == 'CALL'))].copy()

EXCLUDED_UNDERLYING = ['اهرم']
EXCLUDED_NAME_PATTERN = ['1405/04', '1405-04']
exclude_mask = (
    (filter_option['UnderlyingTicker'].isin(EXCLUDED_UNDERLYING)) & 
    (filter_option['Name'].str.contains('|'.join(EXCLUDED_NAME_PATTERN), na=False)))

filter_option = filter_option[~exclude_mask].copy()

# filter_option = filter_option[filter_option['UnderlyingTicker'].isin(EXCLUDED_UNDERLYING)]

from config import EXERCISE_TAX_RATE, get_symbol_market, get_symbol_kind, get_commission_rate, get_exercise_fee_rate  # noqa: E402

results = []
results_fee = []
for underlying_symbol, group in filter_option.groupby('UnderlyingTicker'):
    market = get_symbol_market(underlying_symbol)
    kind = get_symbol_kind(underlying_symbol)

    opt_sell_commission = get_commission_rate(market, 'option', False)
    stock_buy_commission = get_commission_rate(market, kind, True)
    exercise_fee_rate = get_exercise_fee_rate(market, kind)
    exercise_tax_rate = EXERCISE_TAX_RATE

    for index, item in group.iterrows():
        # استخراج اطلاعات مورد نیاز
        ticker = item['Ticker']
        strike_price = item['StrikePrice']
        premium_call = item['BidPrice']
        stock_price = item['UnderlyingPrice']
        contract_size = item['ContractSize']
        days = item['DaysToMaturity']
        
        # محاسبات با کارمزد
        results_with_fees = covered_call_with_fees(
            ticker, premium_call, stock_price, strike_price, contract_size,
            opt_sell_commission, stock_buy_commission,
            exercise_fee_rate, exercise_tax_rate, days)

        # ذخیره نتایج در دیکشنری
        results_fee.append({
            'underlying': underlying_symbol,
            'option_symbol': ticker,
            'strike': strike_price,
            'premium': round(premium_call, 0),
            'stock_price': round(stock_price, 0),
            'net_profit': results_with_fees['net_profit'],
            'monthly_return_%': results_with_fees['monthly_return'],
            'break_even_price': results_with_fees['break_even_price'],
            'max_drop_%': results_with_fees['max_drop_percent'],
            'status': getattr(item['OptionStatus'], 'value', item['OptionStatus']),
            'days_to_maturity': days,
            'volume': int(item.get('Volume', 0))
        })

result_df_fee = pd.DataFrame(results_fee)

In [14]:
# =====================================================================
# سلول نهایی: ادغام نوسانات، محاسبه دلتا و امتیازدهی روی خروجی کارمزدها
# =====================================================================

# --- ۱. فرمول محاسباتی سبک دلتا بلک-شولز ---
def calculate_black_scholes_delta(S, K, T, r, sigma):
    """
    محاسبه دلتای اختیار خرید (Call Option Delta)
    """
    if T <= 0 or sigma <= 0 or S <= 0 or K <= 0:
        return 0.50  # مقدار مرزی فرضی برای آپشن‌های نزدیک سررسید
    
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    return round(float(np.clip(norm.cdf(d1), 0.0, 1.0)), 4)


# --- ۲. تابع امتیازدهی هوشمند هماهنگ با ستون‌های دیتافریم جدید شما ---
def score_covered_call_adapted(row, hv_col='Volatility', delta_col='Delta'):
    """
    سیستم امتیازدهی استراتژی Covered Call منطبق با ساختار ستون‌های سلول قبل
    """
    monthly_return = row['monthly_return_%']
    max_drop = row['max_drop_%']
    dte = row['days_to_maturity']
    stock_price = row['stock_price']
    strike = row['strike']
    hv = row[hv_col]
    delta = row[delta_col]

    # امتیازدهی بخش‌های مختلف
    return_score = np.clip(monthly_return / 12.0, 0.0, 1.0)
    protection_score = np.clip(max_drop / 25.0, 0.0, 1.0)

    # حاشیه امنیت متناسب با نوسان سهم (Expected Move)
    expected_move = 1.5 * hv * np.sqrt(dte / 365.0) * 100
    downside_score = np.clip(max_drop / expected_move, 0.0, 1.0) if expected_move > 0 else 1.0

    # امتیاز جهت‌گیری بازار (Delta Score)
    net_delta = abs(1.0 - delta)
    delta_score = np.exp(-2.0 * net_delta)

    # جریمه نمایی برای گزینه‌های عمیقاً درون سود (ITM)
    moneyness = (stock_price - strike) / stock_price if stock_price > 0 else 0
    itm_penalty = (1.0 - np.exp(-6.0 * moneyness)) if moneyness > 0 else 0.0

    final_score = (
        0.30 * return_score +
        0.25 * protection_score +
        0.25 * downside_score +
        0.10 * delta_score -
        0.10 * itm_penalty)
    return round(np.clip(final_score * 100, 0, 100), 2)


# --- ۳. لود داده‌های نوسان ذخیره شده روزانه ---
vol_file = "daily_market_volatility.xlsx"
if os.path.exists(vol_file):
    df_vol = pd.read_excel(vol_file)
    # هماهنگ‌سازی کلید واژه ادغام با نام ستون سلول شما ('underlying')
    df_vol_subset = df_vol[['UnderlyingTicker', 'Volatility']].rename(columns={'UnderlyingTicker': 'underlying'})
else:
    df_vol_subset = pd.DataFrame(columns=['underlying', 'Volatility'])


# --- ۴. تابع کمکی برای پردازش، محاسبه دلتا و امتیازدهی دیتای ورودی شما ---
def process_and_score_dataset(df_input, df_volatility, r_free=0.25):
    # الف. ادغام با جدول نوسان بر اساس ستون underlying
    df_merged = pd.merge(df_input, df_volatility, on='underlying', how='left')
    df_merged['Volatility'] = df_merged['Volatility'].fillna(0.45) # جایگزینی نوسان‌های خالی با مقدار پیش‌فرض
    
    # ب. محاسبه دلتا به صورت نظیر به نظیر برای تک‌تک آپشن‌ها
    df_merged['Delta'] = df_merged.apply(
        lambda r: calculate_black_scholes_delta(
            S=r['stock_price'],
            K=r['strike'],
            T=r['days_to_maturity'] / 365.0,
            r=r_free,
            sigma=r['Volatility']), axis=1)
    
    # ج. اعمال تابع امتیازدهی
    df_merged['score'] = df_merged.apply(score_covered_call_adapted, axis=1)
    
    # د. مرتب‌سازی صعودی بر اساس بالاترین امتیاز کسب شده
    return df_merged.sort_values(by='score', ascending=False).reset_index(drop=True)


# --- ۵. اجرای خط لوله پردازش برای هر دو حالت (با کارمزد و بدون کارمزد) ---
r_free_rate = 0.25  # نرخ بدون ریسک فرضی

# پردازش دیتافریم محاسبات با کارمزد
scored_df_fee = process_and_score_dataset(result_df_fee, df_vol_subset, r_free=r_free_rate)

# --- 6. نمایش چند ردیف برتر نتایج با اعمال کارمزد جهت بررسی وضعیت نهایی ---
display(
    scored_df_fee[[
        'option_symbol', 'underlying', 'stock_price', 'strike', 
        'days_to_maturity', 'monthly_return_%', 'max_drop_%', 
        'Volatility', 'Delta', 'score'
    ]].head(3))

C:\Users\panahi\AppData\Local\Temp\ipykernel_8804\1327437518.py:69: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_merged['Volatility'] = df_merged['Volatility'].fillna(0.45) # جایگزینی نوسان‌های خالی با مقدار پیش‌فرض


,option_symbol,underlying,stock_price,strike,days_to_maturity,monthly_return_%,max_drop_%,Volatility,Delta,score
0,ضفزر503,فزر,141300,120000,29,5.98,19.69,0.45,0.9342,62.45
1,ضفزر501,فزر,141300,100000,29,2.77,31.06,0.45,0.9984,58.62
2,ضملي7061,فملي,19310,12000,74,2.89,41.97,0.45,0.9965,58.19


In [15]:
scored_df_fee['dte_factor'] = (scored_df_fee['days_to_maturity'] / 30) ** 0.5
scored_df_fee['dte_factor'] = scored_df_fee['dte_factor'].clip(lower=0.3, upper=2.5)

scored_df_fee['max_drop_threshold'] = 10.0 * scored_df_fee['dte_factor']

filtered_df_fee = scored_df_fee[scored_df_fee['max_drop_%'] >= scored_df_fee['max_drop_threshold']].copy()
filtered_df_fee = filtered_df_fee.sort_values(by='monthly_return_%', ascending=False).reset_index(drop=True)
filtered_df_fee.to_excel('filtered_df_fee.xlsx', index=False)